# P05 · Behavior cloning, DAgger and action chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/09_planning_learning/P05_behavior_cloning_dagger_and_action_chunking.ipynb)

**Track:** Planning and learning · **Skill axis (sheet):** Planning and learning · **Time:** about 65 min

**Skill Hub concepts:** Behavior cloning; Action chunking transformer, diffusion policy

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("P05")

## Map: learning a policy from demonstrations

**Behavior cloning (BC):** supervised learning of $\pi_\theta(o) \approx a_{expert}$ on recorded (observation, action)
pairs. Its failure mode is **covariate shift**: the learner makes a small error, reaches states the expert never
visited, has no data there, and errors compound (bound grows like $\epsilon T^2$ over horizon $T$).

**DAgger** (dataset aggregation): roll out the *learner*, ask the expert what it would have done in the states the
learner actually visited, add those labels, retrain. The bound drops to $\epsilon T$.

| Method | Data comes from | Fixes |
|---|---|---|
| BC | expert's own trajectories | nothing about shift |
| DART | expert trajectories with injected noise | some shift, no expert queries online |
| DAgger | learner's trajectories, labeled by the expert | shift, needs an expert you can query |
| ACT (action chunking transformer) | demos; predicts the next $k$ actions at once | compounding over fewer decisions; temporal ensembling smooths |
| Diffusion policy | demos; samples actions by iterative denoising | **multimodal** demos (left or right around an obstacle) that MSE would average into a crash |

**Temporal ensembling (ACT):** at time $t$ you hold several predictions for $a_t$ from chunks made at earlier steps.
Average them with $w_i = e^{-m i}$, where $i = 0$ is the **oldest** prediction, normalized to sum to 1.

Task here: lane keeping. Observation $o = (e, \psi, \kappa)$: lateral error, heading error, road curvature.
The expert demonstrations start exactly on the lane with no disturbance, so they never show what to do when off it.

In [ ]:
import numpy as np
import torch

DT, V = 0.05, 1.0

def kappa(t, phase):
    return 0.5 * np.sin(0.5 * t + phase)

def expert(o):
    o = np.atleast_2d(o)
    return np.clip(V * o[:, 2] - 2.0 * o[:, 0] - 2.5 * o[:, 1], -2, 2)[:, None]

def rollout(policy, phase, noise=0.0, steps=200, rng=None):
    """Return the (steps, 3) observations visited while `policy(o)` drives."""
    e, psi, obs = 0.0, 0.0, []
    for k in range(steps):
        o = np.array([e, psi, kappa(k * DT, phase)])
        obs.append(o)
        u = float(np.clip(np.ravel(policy(o))[0], -2, 2))
        e += V * np.sin(psi) * DT
        psi += (u - V * o[2]) * DT + (rng.normal(0, noise) if noise else 0.0)
    return np.array(obs)

def as_policy(model):
    return lambda o: model(torch.tensor(o, dtype=torch.float32)[None])[0].detach().numpy()

def lane_error(policy, n=8, seed=1):
    r = np.random.default_rng(seed)
    return np.mean([np.mean(np.abs(rollout(policy, ph, 0.03, rng=r)[:, 0])) for ph in r.uniform(0, 2 * np.pi, n)])

rng = np.random.default_rng(0)
X = np.vstack([rollout(lambda o: expert(o)[0], ph) for ph in rng.uniform(0, 2 * np.pi, 5)])
Y = expert(X)
print("demo samples:", X.shape, " expert lane error:", round(lane_error(lambda o: expert(o)[0]), 4))

## Task 1: behavior cloning
Train a multilayer perceptron 3 → 32 → 32 → 1 (tanh activations) with Adam (learning rate 0.01) on full-batch mean
squared error. Call `torch.manual_seed(seed)` first so results repeat.

In [ ]:
def train_bc(X, Y, epochs=400, seed=0):
    raise NotImplementedError  # TODO

lab.check("bc", train_bc)

In [ ]:
try:
    bc = train_bc(X, Y)
    print("behavior cloning lane error:", round(lane_error(as_policy(bc)), 4))
except NotImplementedError:
    print("finish Task 1 first")

## Task 2: one DAgger round
Roll out `policy` once per phase with disturbance `noise`, label every visited observation with `expert`, and return
the aggregated `(X, Y)` (old rows first).

In [ ]:
def dagger_round(policy, X, Y, phases, rng, noise=0.03):
    raise NotImplementedError  # TODO

lab.check("dagger", dagger_round, train_bc)

## Task 3: temporal ensembling

In [ ]:
def temporal_ensemble(preds, m=0.5):
    """preds: list of action arrays predicted for the same time step, oldest first."""
    raise NotImplementedError  # TODO

lab.check("ensemble", temporal_ensemble)

## Checkpoint: Math
**M1 (Apply).** A policy predicts chunks of 10 actions at 50 Hz. Seconds of motion per chunk?

**M2 (Apply).** Temporal ensembling with $m = 0.5$ and 4 predictions: normalized weight of the oldest?

**M3 (Analyze).** BC error bound grows like $\epsilon T^2$ and DAgger like $\epsilon T$. Ratio of the bounds for $T = 100$?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Demonstrators pass an obstacle on the left half the time and on the right the other half. A policy
trained with mean squared error tends to:
- a) pick the left side consistently
- b) average the two and drive into the obstacle
- c) refuse to move

**K2 (Understand).** Explain covariate shift using this lab's numbers (expert, behavior cloning, DAgger lane errors),
and how action chunking reduces the problem without querying an expert.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `p05.h`).
3. In `roboprep/checks/p05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()